# Agente de Mantenimiento Predictivo con MCP HTTP

## Descripción

En este ejercicio, construirás un **servidor MCP propio que expone herramientas vía HTTP** para consultas de mantenimiento predictivo aeronáutico. A diferencia del módulo local (stdio), este servidor es accesible vía HTTP y puede ser consumido por múltiples clientes.

### Arquitectura MCP HTTP
```
Pregunta sobre mantenimiento
    ↓
[MaintenanceAgent]
    ↓
[MCPStreamableHTTPTool] → HTTP (localhost:8000)
    ↓
[Servidor HTTP MCP]
    ↓
[Herramientas HTTP]
    ├→ get_sensor_readings()
    ├→ get_maintenance_history()
    └→ predict_failure_risk()
    ↓
Análisis y respuesta
```

### Ventajas de MCP HTTP
- **Acceso remoto**: El servidor puede estar en una máquina y los clientes en otras
- **Múltiples clientes**: Varios agentes pueden consumir el mismo servidor
- **GitHub Codespaces**: Ideal para desarrollo cloud, con port forwarding
- **Escalabilidad**: Puedes desplegar el servidor en la nube

### Crear el servidor MCP HTTP

Primero, creamos un archivo `http_server.py` que define las herramientas de mantenimiento predictivo usando `FastMCP`. Este servidor se ejecutará con soporte HTTP y será accesible vía HTTP/SSE.

In [ ]:
server_code = """
from mcp.server.fastmcp import FastMCP
from mcp.server.sse import SseServerTransport
from starlette.applications import Starlette
from starlette.routing import Route
import uvicorn
import asyncio

# Crear instancia de FastMCP
mcp = FastMCP("AeroMaintenance")

@mcp.tool()
def get_sensor_readings(component: str) -> dict:
    \"\"\"Devuelve las lecturas de sensores para un componente aeronáutico específico.
    
    Args:
        component: Nombre del componente (ej: 'turbina', 'compresor', 'rodamiento')
    
    Returns:
        dict: Lecturas de temperatura, vibración y presión
    \"\"\"
    sensor_data = {
        "turbina": {
            "temperatura_celsius": 850,
            "vibracion_mm_s": 4.2,
            "presion_bar": 12.5,
            "horas_operacion": 2847
        },
        "compresor": {
            "temperatura_celsius": 420,
            "vibracion_mm_s": 2.8,
            "presion_bar": 8.3,
            "horas_operacion": 3120
        },
        "rodamiento": {
            "temperatura_celsius": 95,
            "vibracion_mm_s": 6.7,
            "presion_bar": 2.1,
            "horas_operacion": 4890
        },
        "combustor": {
            "temperatura_celsius": 1200,
            "vibracion_mm_s": 1.9,
            "presion_bar": 15.8,
            "horas_operacion": 1950
        }
    }
    return sensor_data.get(component.lower(), {"error": "Componente no encontrado"})

@mcp.tool()
def get_maintenance_history(component: str) -> dict:
    \"\"\"Devuelve el historial de mantenimiento de un componente.
    
    Args:
        component: Nombre del componente
    
    Returns:
        dict: Historial de mantenimientos realizados
    \"\"\"
    maintenance_data = {
        "turbina": {
            "ultimo_mantenimiento": "2024-09-15",
            "tipo": "Inspección mayor",
            "proximo_programado": "2024-12-15",
            "mantenimientos_totales": 12
        },
        "compresor": {
            "ultimo_mantenimiento": "2024-08-20",
            "tipo": "Limpieza y reemplazo de filtros",
            "proximo_programado": "2024-11-20",
            "mantenimientos_totales": 18
        },
        "rodamiento": {
            "ultimo_mantenimiento": "2024-06-10",
            "tipo": "Lubricación",
            "proximo_programado": "2024-12-10",
            "mantenimientos_totales": 24
        },
        "combustor": {
            "ultimo_mantenimiento": "2024-10-05",
            "tipo": "Inspección de boquillas",
            "proximo_programado": "2025-01-05",
            "mantenimientos_totales": 8
        }
    }
    return maintenance_data.get(component.lower(), {"error": "Componente no encontrado"})

@mcp.tool()
def predict_failure_risk(component: str) -> dict:
    \"\"\"Predice el riesgo de falla basado en sensores y mantenimiento.
    
    Args:
        component: Nombre del componente
    
    Returns:
        dict: Nivel de riesgo y recomendaciones
    \"\"\"
    risk_analysis = {
        "turbina": {
            "nivel_riesgo": "MEDIO",
            "probabilidad_falla": "35%",
            "tiempo_estimado_falla": "45-60 días",
            "recomendacion": "Programar inspección en las próximas 2 semanas",
            "factores": ["Temperatura elevada", "Horas de operación altas"]
        },
        "compresor": {
            "nivel_riesgo": "BAJO",
            "probabilidad_falla": "12%",
            "tiempo_estimado_falla": ">90 días",
            "recomendacion": "Mantenimiento de rutina según calendario",
            "factores": ["Parámetros dentro de rango normal"]
        },
        "rodamiento": {
            "nivel_riesgo": "ALTO",
            "probabilidad_falla": "68%",
            "tiempo_estimado_falla": "15-30 días",
            "recomendacion": "URGENTE: Reemplazar rodamiento inmediatamente",
            "factores": ["Vibración excesiva", "Horas de operación críticas", "Mantenimiento atrasado"]
        },
        "combustor": {
            "nivel_riesgo": "BAJO",
            "probabilidad_falla": "8%",
            "tiempo_estimado_falla": ">120 días",
            "recomendacion": "Componente en excelente estado",
            "factores": ["Mantenimiento reciente", "Baja vibración"]
        }
    }
    return risk_analysis.get(component.lower(), {"error": "Componente no encontrado"})

# Configurar el servidor HTTP con SSE
async def handle_sse(request):
    async with SseServerTransport("/messages") as transport:
        await mcp.run(transport)

# Crear la aplicación Starlette
app = Starlette(
    routes=[
        Route("/sse", endpoint=handle_sse),
    ]
)

if __name__ == "__main__":
    print("🚀 Starting MCP HTTP Server...")
    print("📡 Server will be available at: http://localhost:8000/sse")
    print("💡 In GitHub Codespaces, use port forwarding to access this server")
    uvicorn.run(app, host="0.0.0.0", port=8000)
\"\"\"

print("Creating HTTP MCP server file...")
with open("http_mcp_server.py", "w") as f:
    f.write(server_code)
print("✅ Server file 'http_mcp_server.py' created successfully")

### Iniciar el servidor MCP HTTP en segundo plano

**IMPORTANTE para GitHub Codespaces:**
1. Ejecuta esta celda para iniciar el servidor en segundo plano
2. Espera a que veas el mensaje "Server will be available at..."
3. En Codespaces, ve a la pestaña "PORTS" y haz público el puerto 8000
4. Copia la URL pública (será algo como `https://xxx-8000.app.github.dev`)

In [ ]:
import subprocess
import time
import os
import signal

# Terminar proceso anterior si existe
try:
    with open(".mcp_server_pid", "r") as f:
        old_pid = int(f.read())
        os.kill(old_pid, signal.SIGTERM)
        print(f"✅ Servidor anterior (PID {old_pid}) terminado")
except:
    pass

# Iniciar el servidor en segundo plano
print("🚀 Iniciando servidor HTTP MCP...")
process = subprocess.Popen(
    ["python", "http_mcp_server.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Guardar PID para poder terminarlo después
with open(".mcp_server_pid", "w") as f:
    f.write(str(process.pid))

print(f"✅ Servidor iniciado con PID: {process.pid}")
print("⏳ Esperando 3 segundos para que el servidor se inicie...")
time.sleep(3)

print("\n📡 Servidor HTTP MCP corriendo en http://localhost:8000/sse")
print("\n💡 Para GitHub Codespaces:")
print("   1. Ve a la pestaña 'PORTS' en VS Code")
print("   2. Busca el puerto 8000")
print("   3. Haz clic derecho → 'Port Visibility' → 'Public'")
print("   4. Copia la URL del puerto 8000")
print("   5. Usa esa URL en el cliente (siguiente celda)\n")

### Cargar librerías y variables de entorno para el cliente

In [ ]:
import asyncio
import os
from dotenv import load_dotenv, find_dotenv
from agent_framework import ChatAgent, MCPStreamableHTTPTool
from agent_framework.openai import OpenAIChatClient

load_dotenv(find_dotenv(usecwd=True))

# GitHub Models client configuration
MODEL_NAME = os.getenv("GITHUB_MODEL", "openai/gpt-4o")

print("✅ Librerías cargadas correctamente")

### Crear y ejecutar el agente cliente

El `ChatAgent` se conecta al servidor MCP HTTP usando `MCPStreamableHTTPTool`. 

**Configuración de URL:**
- **Local**: `http://localhost:8000/sse`
- **Codespaces**: Usa la URL pública del puerto 8000 (ej: `https://xxx-8000.app.github.dev/sse`)

In [ ]:
async def query_maintenance(prompt, server_url="http://localhost:8000/sse"):
    """
    Consulta el sistema de mantenimiento predictivo.
    
    Args:
        prompt: Pregunta sobre mantenimiento
        server_url: URL del servidor MCP HTTP
                   - Local: "http://localhost:8000/sse"
                   - Codespaces: "https://xxx-8000.app.github.dev/sse"
    """
    print(f"🔌 Conectando al servidor MCP HTTP: {server_url}")
    
    # GitHub Models client configuration
    client = OpenAIChatClient(
        model_id=MODEL_NAME,
        api_key=os.environ["GITHUB_TOKEN"],
        base_url="https://models.github.ai/inference"
    )
    
    async with (
        MCPStreamableHTTPTool(
            name="aeromaintenance",
            url=server_url,
            # No se requiere Authorization para servidor local
            # Para producción, puedes agregar: headers={"Authorization": "Bearer token"}
        ) as mcp_server,
        ChatAgent(
            chat_client=client,
            name="MaintenanceAgent",
            instructions=(
                "Eres un asistente de mantenimiento predictivo aeronáutico que analiza "
                "lecturas de sensores, historial de mantenimiento y predice riesgos de falla. "
                "Proporciona análisis detallados y recomendaciones claras basadas en los datos."
            ),
        ) as agent,
    ):
        print("✅ Conexión establecida con el servidor MCP HTTP")
        print(f"\n❓ Ejecutando consulta: {prompt}\n")
        result = await agent.run(prompt, tools=mcp_server)
        print("\n📊 Resultado de la consulta:")
        print("=" * 80)
        print(result)
        print("=" * 80)

### Ejemplo 1: Análisis de riesgo de múltiples componentes

In [ ]:
# Cambia esta URL si estás en Codespaces
SERVER_URL = "http://localhost:8000/sse"
# Para Codespaces, usa algo como:
# SERVER_URL = "https://your-codespace-8000.app.github.dev/sse"

print("🔍 Análisis de componentes críticos...\n")
user_prompt = "Analiza el riesgo de falla de la turbina y el rodamiento. ¿Cuál requiere atención urgente?"
await query_maintenance(user_prompt, SERVER_URL)

### Ejemplo 2: Planificación de mantenimiento

In [ ]:
SERVER_URL = "http://localhost:8000/sse"

print("📅 Planificación de mantenimiento...\n")
user_prompt = "Crea un plan de mantenimiento para las próximas 4 semanas basado en los riesgos y el historial de todos los componentes."
await query_maintenance(user_prompt, SERVER_URL)

### Ejemplo 3: Diagnóstico de componente específico

In [ ]:
SERVER_URL = "http://localhost:8000/sse"

print("🔧 Diagnóstico de componente específico...\n")
user_prompt = "Dame un diagnóstico completo del compresor: sensores, historial y predicción de falla."
await query_maintenance(user_prompt, SERVER_URL)

### Detener el servidor (ejecuta al final)

Ejecuta esta celda cuando termines de usar el servidor.

In [ ]:
import os
import signal

try:
    with open(".mcp_server_pid", "r") as f:
        pid = int(f.read())
        os.kill(pid, signal.SIGTERM)
        print(f"✅ Servidor HTTP MCP (PID {pid}) detenido correctamente")
    os.remove(".mcp_server_pid")
except FileNotFoundError:
    print("⚠️  No se encontró un servidor en ejecución")
except Exception as e:
    print(f"❌ Error al detener el servidor: {e}")

## Resumen

### Lo que aprendiste:

1. **Crear un servidor MCP HTTP** con FastMCP y SSE (Server-Sent Events)
2. **Exponer herramientas personalizadas** mediante endpoints HTTP
3. **Conectar un agente cliente** al servidor MCP HTTP usando `MCPStreamableHTTPTool`
4. **Trabajar en GitHub Codespaces** con port forwarding para desarrollo cloud

### Diferencias clave vs MCP Local (Módulo 07):

| Aspecto | MCP Local (stdio) | MCP HTTP |
|---------|-------------------|----------|
| **Transporte** | stdio (stdin/stdout) | HTTP con SSE |
| **Tool** | `MCPStdioTool` | `MCPStreamableHTTPTool` |
| **Acceso** | Solo proceso local | Red (localhost o remoto) |
| **Clientes** | Uno a la vez | Múltiples simultáneos |
| **Despliegue** | No aplicable | Puede desplegarse en cloud |
| **Port forwarding** | No necesario | Necesario en Codespaces |

### Casos de uso ideales para MCP HTTP:

- ✅ Múltiples agentes consumiendo el mismo servidor
- ✅ Desarrollo en GitHub Codespaces o entornos cloud
- ✅ Herramientas que necesitan ser accesibles desde diferentes máquinas
- ✅ Microservicios y arquitecturas distribuidas
- ✅ APIs de herramientas que otros equipos pueden consumir

### Próximos pasos:

- 🚀 Despliega tu servidor MCP en un servicio cloud (Azure Container Apps, AWS ECS, etc.)
- 🔐 Añade autenticación con tokens JWT o API keys
- 📊 Agrega logging y monitoreo con Application Insights
- 🔄 Implementa balanceo de carga para alta disponibilidad